# The file will build Random Forest Regression and KNN and Neural Network Models.

## a). Random Forest Regression

### 1. Load the data and collect features can be used for prediction

In [1]:
# import needed libraries
import numpy as np
import pandas as pd
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers
from sklearn.metrics import r2_score
import statsmodels.api as sm # pip install statsmodels
from sklearn.model_selection import train_test_split
from sklearn.model_selection import GridSearchCV
from sklearn.preprocessing import StandardScaler
from sklearn.ensemble import RandomForestRegressor
from sklearn.neighbors import KNeighborsRegressor
from sklearn.metrics import mean_squared_error, mean_absolute_error

2023-10-04 16:40:36.629731: I tensorflow/tsl/cuda/cudart_stub.cc:28] Could not find cuda drivers on your machine, GPU will not be used.
2023-10-04 16:40:37.006193: I tensorflow/tsl/cuda/cudart_stub.cc:28] Could not find cuda drivers on your machine, GPU will not be used.
2023-10-04 16:40:37.009059: I tensorflow/core/platform/cpu_feature_guard.cc:182] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
2023-10-04 16:40:38.637226: W tensorflow/compiler/tf2tensorrt/utils/py_utils.cc:38] TF-TRT Warning: Could not find TensorRT


In [2]:
modelling_data = pd.read_csv('../data/curated/final_data/modelling_data_2023.csv')
modelling_data

,personal_income,pop_density,offence_count,log_school_distance,log_station_distance,log_hospital_distance,log_mall_distance,log_park_distance,log_CBD_distance,num_bedroom,num_bathroom,rental_price
0,72706.625309,965.216952,17.333333,0.640804,2.475412,1.906040,1.239505,1.345502,4.578730,4,2,575.0
1,72706.625309,1746.760558,17.333333,-0.348952,1.681479,1.438925,1.052926,1.716843,4.559835,4,2,560.0
2,72706.625309,965.216952,17.333333,0.095820,2.187960,1.847653,1.673884,1.622239,4.586220,2,2,490.0
3,72706.625309,965.216952,17.333333,-0.393160,2.092679,1.736704,0.793410,1.183953,4.578058,4,2,540.0
4,70490.898811,151.921192,31.000000,0.799813,2.315778,2.017863,1.375422,0.115940,4.602687,4,2,520.0
...,...,...,...,...,...,...,...,...,...,...,...,...
8214,100462.815420,1773.237842,25.000000,-2.228826,0.138021,0.374363,0.344505,1.026702,3.826546,2,1,630.0
8215,100462.815420,1773.237842,25.000000,-1.427862,0.754430,1.122437,0.873918,1.320736,3.823240,4,3,730.0
8216,100462.815420,1773.237842,25.000000,-0.199202,1.034856,1.025788,0.821525,1.484705,3.805227,3,1,450.0
8217,100462.815420,1773.237842,25.000000,-1.237029,0.614537,0.623725,0.746158,1.316399,3.810149,1,1,300.0


In [3]:
# collect all features
final_features = modelling_data.columns
features_list = final_features.tolist()
features_list.remove('rental_price')
features_list

['personal_income',
 'pop_density',
 'offence_count',
 'log_school_distance',
 'log_station_distance',
 'log_hospital_distance',
 'log_mall_distance',
 'log_park_distance',
 'log_CBD_distance',
 'num_bedroom',
 'num_bathroom']

###  2. Splite the whole data to train & test data --- and build random forest regression

In [4]:
# split training set and test set
X_train, X_test, y_train, y_test = train_test_split(modelling_data[features_list], modelling_data['rental_price'], test_size = 0.1, random_state=123)

# create random forest model
rf_model = RandomForestRegressor(random_state=123)

# define hyperparameter ranges
param_grid = {
    'n_estimators': [100, 150, 200],
    'max_depth': [None, 10, 20, 30],
    'min_samples_split': [2, 5, 10]
}

# create Grid Search object
grid_search = GridSearchCV(estimator=rf_model, param_grid=param_grid, cv=5, scoring='neg_mean_squared_error', n_jobs=-1)

# run a Grid Search to find the best hyperparameter combination
grid_search.fit(X_train, y_train)

# print optimal hyperparameter combinations and performance evaluation
print("Best Hyperparameters: ", grid_search.best_params_)
print("Best Negative Mean Squared Error: ", grid_search.best_score_)

Best Hyperparameters:  {'max_depth': 20, 'min_samples_split': 5, 'n_estimators': 150}
Best Negative Mean Squared Error:  -7750.404347753676


In [5]:
# create random rorest model using optimal hyperparameters
best_n_estimators = grid_search.best_params_['n_estimators']
best_max_depth = grid_search.best_params_['max_depth']
best_min_samples_split = grid_search.best_params_['min_samples_split']

final_rf_model = RandomForestRegressor(n_estimators=best_n_estimators, max_depth=best_max_depth, min_samples_split=best_min_samples_split, random_state=123)

# train the final model
final_rf_model.fit(X_train, y_train)

# make predictions and evaluate the model
y_pred = final_rf_model.predict(X_test)
mse = mean_squared_error(y_test, y_pred)
mae = mean_absolute_error(y_test, y_pred)
r2 = r2_score(y_test, y_pred)
print("MSE: {:.2f}".format(mse))
print("MAE: {:.2f}".format(mae))
print("r2 score: {:.2f}".format(r2))

MSE: 6602.44
MAE: 55.13
r2 score: 0.72


## b). KNN Model

In [6]:
# split training set and test set
X_train, X_test, y_train, y_test = train_test_split(modelling_data[features_list], modelling_data['rental_price'], test_size = 0.1, random_state=123)

# scale the data with features
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

# try different K values, to find the best k with lowest mse
best_mse = float('inf')
best_k = None
for k in range(1, 21):
    knn = KNeighborsRegressor(n_neighbors=k, weights='distance', p=2, metric='minkowski')
    knn.fit(X_train_scaled, y_train)
    y_pred = knn.predict(X_test_scaled)
    mse = mean_squared_error(y_test, y_pred)
    if mse < best_mse:
        best_mse = mse
        best_k = k
print('Best K value is: {:.2f}'.format(best_k))

# train KNN model
best_knn = KNeighborsRegressor(n_neighbors=best_k, weights='distance', p=2, metric='minkowski')
best_knn.fit(X_train_scaled, y_train)

# make predictions and evaluate the model
y_pred = best_knn.predict(X_test_scaled)
mse = mean_squared_error(y_test, y_pred)
mae = mean_absolute_error(y_test, y_pred)
r2 = r2_score(y_test, y_pred)
print("MSE: {:.2f}".format(mse))
print("MAE: {:.2f}".format(mae))
print("r2 score: {:.2f}".format(r2))

Best K value is: 12.00
MSE: 8391.34
MAE: 61.30
r2 score: 0.65


## c). Neural Network Model

In [7]:
# create a sequential model
model = keras.Sequential()

# add input layer, hidden layer, add output layer
model.add(layers.Input(shape=(11,)))  # 11 is the num of features

model.add(layers.Dense(64, activation='relu'))
model.add(layers.Dense(32, activation='relu'))

model.add(layers.Dense(1))

# build and train the model
model.compile(optimizer='adam', loss='mean_squared_error', metrics=['mae', 'mse'])
model.summary()
history = model.fit(X_train_scaled, y_train, epochs=100, validation_split=0.2, verbose=2)

# # make predictions and evaluate the model
y_pred = model.predict(X_test_scaled)
mse = mean_squared_error(y_test, y_pred)
mae = mean_absolute_error(y_test, y_pred)
r2 = r2_score(y_test, y_pred)
print("MSE: {:.2f}".format(mse))
print("MAE: {:.2f}".format(mae))
print("r2 score: {:.2f}".format(r2))

Model: "sequential"
_________________________________________________________________
 Layer (type)                Output Shape              Param #   
 dense (Dense)               (None, 64)                768       
                                                                 
 dense_1 (Dense)             (None, 32)                2080      
                                                                 
 dense_2 (Dense)             (None, 1)                 33        
                                                                 
Total params: 2881 (11.25 KB)
Trainable params: 2881 (11.25 KB)
Non-trainable params: 0 (0.00 Byte)
_________________________________________________________________
Epoch 1/100


2023-10-04 16:43:14.841417: I tensorflow/compiler/xla/stream_executor/cuda/cuda_gpu_executor.cc:981] could not open file to read NUMA node: /sys/bus/pci/devices/0000:01:00.0/numa_node
Your kernel may have been built without NUMA support.
2023-10-04 16:43:14.841909: W tensorflow/core/common_runtime/gpu/gpu_device.cc:1960] Cannot dlopen some GPU libraries. Please make sure the missing libraries mentioned above are installed properly if you would like to use GPU. Follow the guide at https://www.tensorflow.org/install/gpu for how to download and setup the required libraries for your platform.
Skipping registering GPU devices...


185/185 - 1s - loss: 294786.4375 - mae: 518.6917 - mse: 294786.4375 - val_loss: 211691.2188 - val_mae: 432.8056 - val_mse: 211691.2188 - 1s/epoch - 6ms/step
Epoch 2/100
185/185 - 0s - loss: 112453.3828 - mae: 284.0973 - mse: 112453.3828 - val_loss: 53928.4922 - val_mae: 186.9625 - val_mse: 53928.4922 - 326ms/epoch - 2ms/step
Epoch 3/100
185/185 - 0s - loss: 45321.0000 - mae: 166.3099 - mse: 45321.0000 - val_loss: 35877.6484 - val_mae: 146.9787 - val_mse: 35877.6484 - 318ms/epoch - 2ms/step
Epoch 4/100
185/185 - 0s - loss: 33902.1406 - mae: 138.0413 - mse: 33902.1406 - val_loss: 28369.3594 - val_mae: 127.8025 - val_mse: 28369.3594 - 344ms/epoch - 2ms/step
Epoch 5/100
185/185 - 0s - loss: 28385.4004 - mae: 122.8318 - mse: 28385.4004 - val_loss: 24268.3047 - val_mae: 117.1023 - val_mse: 24268.3047 - 316ms/epoch - 2ms/step
Epoch 6/100
185/185 - 0s - loss: 25058.6660 - mae: 113.8319 - mse: 25058.6660 - val_loss: 21741.1953 - val_mae: 110.3381 - val_mse: 21741.1953 - 320ms/epoch - 2ms/step
E

# *Finding: RF regression model has the best performance.

### 1. predict price 3 years later, and calculate the growth rate

In [8]:
# select needed features for prediction of 2026 rental price
data_2026 = pd.read_csv("../data/curated/final_data/merged_data_2026.csv")
data_2026_test = data_2026[features_list]
data_2026_test

,personal_income,pop_density,offence_count,log_school_distance,log_station_distance,log_hospital_distance,log_mall_distance,log_park_distance,log_CBD_distance,num_bedroom,num_bathroom
0,79514.553071,2814.014080,23.132872,0.640804,2.475412,1.906040,1.239505,1.345502,4.578730,4,2
1,79514.553071,4306.277854,23.132872,-0.348952,1.681479,1.438925,1.052926,1.716843,4.559835,4,2
2,79514.553071,2814.014080,23.132872,0.095820,2.187960,1.847653,1.673884,1.622239,4.586220,2,2
3,79514.553071,2814.014080,23.132872,-0.393160,2.092679,1.736704,0.793410,1.183953,4.578058,4,2
4,75319.429183,184.687628,34.708923,0.799813,2.315778,2.017863,1.375422,0.115940,4.602687,4,2
...,...,...,...,...,...,...,...,...,...,...,...
8214,112895.360426,1832.451409,25.631382,-2.228826,0.138021,0.374363,0.344505,1.026702,3.826546,2,1
8215,112895.360426,1832.451409,25.631382,-1.427862,0.754430,1.122437,0.873918,1.320736,3.823240,4,3
8216,112895.360426,1832.451409,25.631382,-0.199202,1.034856,1.025788,0.821525,1.484705,3.805227,3,1
8217,112895.360426,1832.451409,25.631382,-1.237029,0.614537,0.623725,0.746158,1.316399,3.810149,1,1


In [9]:
# predict the 2026 rental price by RF
pred_2026 = final_rf_model.predict(data_2026_test)
pred_2026

array([636.01378307, 626.71496032, 566.54353415, ..., 578.28616643,
       351.03117401, 409.6981394 ])

In [10]:
# save the prediction as a new row of data, and record its 2023 price
data_2023 = pd.read_csv("../data/curated/final_data/merged_data_2023.csv") 
data_2026['rental_price_2023'] = data_2023['rental_price']
data_2026['predicted_price_2026'] = pred_2026
data_2026

,name,coordinates,sa2_code,sa2_name,sa2_geometry,num_bedroom,num_bathroom,num_parking,pop_density,personal_income,offence_count,log_school_distance,log_hospital_distance,log_mall_distance,log_park_distance,log_station_distance,log_CBD_distance,rental_price_2023,predicted_price_2026
0,31 Chittagong Drive Clyde North VIC 3978,"[-38.1053122, 145.3570863]",212031556.0,Clyde North - South,POLYGON ((145.37034567475234 -38.0937851973191...,4,2,2.0,2814.014080,79514.553071,23.132872,0.640804,1.906040,1.239505,1.345502,2.475412,4.578730,575.0,636.013783
1,50 Elmtree Crescent Clyde North VIC 3978,"[-38.0825712, 145.3561984]",212031555.0,Clyde North - North,POLYGON ((145.3346630127421 -38.07820964472685...,4,2,2.0,4306.277854,79514.553071,23.132872,-0.348952,1.438925,1.052926,1.716843,1.681479,4.559835,560.0,626.714960
2,7 Mortdale Lane Clyde North VIC 3978,"[-38.0961758, 145.3800644]",212031556.0,Clyde North - South,POLYGON ((145.37034567475234 -38.0937851973191...,2,2,1.0,2814.014080,79514.553071,23.132872,0.095820,1.847653,1.673884,1.622239,2.187960,4.586220,490.0,566.543534
3,54 Walhallow Drive Clyde North VIC 3978,"[-38.1133324, 145.3457396]",212031556.0,Clyde North - South,POLYGON ((145.37034567475234 -38.0937851973191...,4,2,1.0,2814.014080,79514.553071,23.132872,-0.393160,1.736704,0.793410,1.183953,2.092679,4.578058,540.0,586.459791
4,10 Sicily Road Clyde North VIC 3978,"[-38.1295789, 145.3642993]",212031303.0,Cranbourne South,POLYGON ((145.3254642788471 -38.12742071438774...,4,2,2.0,184.687628,75319.429183,34.708923,0.799813,2.017863,1.375422,0.115940,2.315778,4.602687,520.0,593.995740
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
8214,61 Tongue Street Yarraville VIC 3013,"[-37.8131463, 144.8909053]",213031352.0,Yarraville,POLYGON ((144.85914995429522 -37.8176431203511...,2,1,0.0,1832.451409,112895.360426,25.631382,-2.228826,0.374363,0.344505,1.026702,0.138021,3.826546,630.0,613.106051
8215,47 Mill Avenue Yarraville VIC 3013,"[-37.8222822, 144.872198]",213031352.0,Yarraville,POLYGON ((144.85914995429522 -37.8176431203511...,4,3,2.0,1832.451409,112895.360426,25.631382,-1.427862,1.122437,0.873918,1.320736,0.754430,3.823240,730.0,781.694406
8216,12 Adeney Street Yarraville VIC 3013,"[-37.8163817, 144.8666543]",213031352.0,Yarraville,POLYGON ((144.85914995429522 -37.8176431203511...,3,1,2.0,1832.451409,112895.360426,25.631382,-0.199202,1.025788,0.821525,1.484705,1.034856,3.805227,450.0,578.286166
8217,229B Somerville Road Yarraville VIC 3013,"[-37.8124289, 144.8779569]",213031352.0,Yarraville,POLYGON ((144.85914995429522 -37.8176431203511...,1,1,0.0,1832.451409,112895.360426,25.631382,-1.237029,0.623725,0.746158,1.316399,0.614537,3.810149,300.0,351.031174


In [11]:
# calculate rental price growth_rate(%) for properties(2023~2026)
data_2026['growth_rate(%)'] = ((data_2026['predicted_price_2026'] - data_2026['rental_price_2023']) / data_2026['rental_price_2023']) * 100
data_2026

,name,coordinates,sa2_code,sa2_name,sa2_geometry,num_bedroom,num_bathroom,num_parking,pop_density,personal_income,offence_count,log_school_distance,log_hospital_distance,log_mall_distance,log_park_distance,log_station_distance,log_CBD_distance,rental_price_2023,predicted_price_2026,growth_rate(%)
0,31 Chittagong Drive Clyde North VIC 3978,"[-38.1053122, 145.3570863]",212031556.0,Clyde North - South,POLYGON ((145.37034567475234 -38.0937851973191...,4,2,2.0,2814.014080,79514.553071,23.132872,0.640804,1.906040,1.239505,1.345502,2.475412,4.578730,575.0,636.013783,10.611093
1,50 Elmtree Crescent Clyde North VIC 3978,"[-38.0825712, 145.3561984]",212031555.0,Clyde North - North,POLYGON ((145.3346630127421 -38.07820964472685...,4,2,2.0,4306.277854,79514.553071,23.132872,-0.348952,1.438925,1.052926,1.716843,1.681479,4.559835,560.0,626.714960,11.913386
2,7 Mortdale Lane Clyde North VIC 3978,"[-38.0961758, 145.3800644]",212031556.0,Clyde North - South,POLYGON ((145.37034567475234 -38.0937851973191...,2,2,1.0,2814.014080,79514.553071,23.132872,0.095820,1.847653,1.673884,1.622239,2.187960,4.586220,490.0,566.543534,15.621129
3,54 Walhallow Drive Clyde North VIC 3978,"[-38.1133324, 145.3457396]",212031556.0,Clyde North - South,POLYGON ((145.37034567475234 -38.0937851973191...,4,2,1.0,2814.014080,79514.553071,23.132872,-0.393160,1.736704,0.793410,1.183953,2.092679,4.578058,540.0,586.459791,8.603665
4,10 Sicily Road Clyde North VIC 3978,"[-38.1295789, 145.3642993]",212031303.0,Cranbourne South,POLYGON ((145.3254642788471 -38.12742071438774...,4,2,2.0,184.687628,75319.429183,34.708923,0.799813,2.017863,1.375422,0.115940,2.315778,4.602687,520.0,593.995740,14.229950
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
8214,61 Tongue Street Yarraville VIC 3013,"[-37.8131463, 144.8909053]",213031352.0,Yarraville,POLYGON ((144.85914995429522 -37.8176431203511...,2,1,0.0,1832.451409,112895.360426,25.631382,-2.228826,0.374363,0.344505,1.026702,0.138021,3.826546,630.0,613.106051,-2.681579
8215,47 Mill Avenue Yarraville VIC 3013,"[-37.8222822, 144.872198]",213031352.0,Yarraville,POLYGON ((144.85914995429522 -37.8176431203511...,4,3,2.0,1832.451409,112895.360426,25.631382,-1.427862,1.122437,0.873918,1.320736,0.754430,3.823240,730.0,781.694406,7.081425
8216,12 Adeney Street Yarraville VIC 3013,"[-37.8163817, 144.8666543]",213031352.0,Yarraville,POLYGON ((144.85914995429522 -37.8176431203511...,3,1,2.0,1832.451409,112895.360426,25.631382,-0.199202,1.025788,0.821525,1.484705,1.034856,3.805227,450.0,578.286166,28.508037
8217,229B Somerville Road Yarraville VIC 3013,"[-37.8124289, 144.8779569]",213031352.0,Yarraville,POLYGON ((144.85914995429522 -37.8176431203511...,1,1,0.0,1832.451409,112895.360426,25.631382,-1.237029,0.623725,0.746158,1.316399,0.614537,3.810149,300.0,351.031174,17.010391


### 2. save the prediction results as a CSV file

In [12]:
path = '../data/curated/final_data/predicted_data_2026.csv'
data_2026.to_csv(path, index=False)
data_2026 = pd.read_csv("../data/curated/final_data/predicted_data_2026.csv") 
data_2026

,name,coordinates,sa2_code,sa2_name,sa2_geometry,num_bedroom,num_bathroom,num_parking,pop_density,personal_income,offence_count,log_school_distance,log_hospital_distance,log_mall_distance,log_park_distance,log_station_distance,log_CBD_distance,rental_price_2023,predicted_price_2026,growth_rate(%)
0,31 Chittagong Drive Clyde North VIC 3978,"[-38.1053122, 145.3570863]",212031556.0,Clyde North - South,POLYGON ((145.37034567475234 -38.0937851973191...,4,2,2.0,2814.014080,79514.553071,23.132872,0.640804,1.906040,1.239505,1.345502,2.475412,4.578730,575.0,636.013783,10.611093
1,50 Elmtree Crescent Clyde North VIC 3978,"[-38.0825712, 145.3561984]",212031555.0,Clyde North - North,POLYGON ((145.3346630127421 -38.07820964472685...,4,2,2.0,4306.277854,79514.553071,23.132872,-0.348952,1.438925,1.052926,1.716843,1.681479,4.559835,560.0,626.714960,11.913386
2,7 Mortdale Lane Clyde North VIC 3978,"[-38.0961758, 145.3800644]",212031556.0,Clyde North - South,POLYGON ((145.37034567475234 -38.0937851973191...,2,2,1.0,2814.014080,79514.553071,23.132872,0.095820,1.847653,1.673884,1.622239,2.187960,4.586220,490.0,566.543534,15.621129
3,54 Walhallow Drive Clyde North VIC 3978,"[-38.1133324, 145.3457396]",212031556.0,Clyde North - South,POLYGON ((145.37034567475234 -38.0937851973191...,4,2,1.0,2814.014080,79514.553071,23.132872,-0.393160,1.736704,0.793410,1.183953,2.092679,4.578058,540.0,586.459791,8.603665
4,10 Sicily Road Clyde North VIC 3978,"[-38.1295789, 145.3642993]",212031303.0,Cranbourne South,POLYGON ((145.3254642788471 -38.12742071438774...,4,2,2.0,184.687628,75319.429183,34.708923,0.799813,2.017863,1.375422,0.115940,2.315778,4.602687,520.0,593.995740,14.229950
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
8214,61 Tongue Street Yarraville VIC 3013,"[-37.8131463, 144.8909053]",213031352.0,Yarraville,POLYGON ((144.85914995429522 -37.8176431203511...,2,1,0.0,1832.451409,112895.360426,25.631382,-2.228826,0.374363,0.344505,1.026702,0.138021,3.826546,630.0,613.106051,-2.681579
8215,47 Mill Avenue Yarraville VIC 3013,"[-37.8222822, 144.872198]",213031352.0,Yarraville,POLYGON ((144.85914995429522 -37.8176431203511...,4,3,2.0,1832.451409,112895.360426,25.631382,-1.427862,1.122437,0.873918,1.320736,0.754430,3.823240,730.0,781.694406,7.081425
8216,12 Adeney Street Yarraville VIC 3013,"[-37.8163817, 144.8666543]",213031352.0,Yarraville,POLYGON ((144.85914995429522 -37.8176431203511...,3,1,2.0,1832.451409,112895.360426,25.631382,-0.199202,1.025788,0.821525,1.484705,1.034856,3.805227,450.0,578.286166,28.508037
8217,229B Somerville Road Yarraville VIC 3013,"[-37.8124289, 144.8779569]",213031352.0,Yarraville,POLYGON ((144.85914995429522 -37.8176431203511...,1,1,0.0,1832.451409,112895.360426,25.631382,-1.237029,0.623725,0.746158,1.316399,0.614537,3.810149,300.0,351.031174,17.010391


In [13]:
# calculate average growth_rate(%) for each sa2_name
average_growth_rate_by_sa2 = data_2026.groupby('sa2_name')['growth_rate(%)'].mean().reset_index()
print(average_growth_rate_by_sa2)

                       sa2_name  growth_rate(%)
0                    Abbotsford        4.902311
1                  Airport West        4.133877
2                   Albert Park        0.855093
3                     Alfredton       20.628217
4        Alphington - Fairfield        7.634165
..                          ...             ...
350       Wonthaggi - Inverloch        7.155537
351        Wyndham Vale - North       17.468025
352        Wyndham Vale - South       22.369693
353  Yallourn North - Glengarry        0.787844
354                  Yarraville        3.959479

[355 rows x 2 columns]


In [14]:
# sort the average growth rate for rental price, and find the top 10 sa2 areas with highest value
sorted_growth_rates = average_growth_rate_by_sa2.sort_values(by='growth_rate(%)', ascending=False)
top_10_sa2 = sorted_growth_rates.head(10)
print(top_10_sa2)

                   sa2_name  growth_rate(%)
218              Myrtleford      242.936746
111          Doreen - North       42.450243
112          Doreen - South       37.311957
254      Point Cook - South       34.572889
208        Moe - Newborough       30.490874
279                Seabrook       26.918311
265  Reservoir - North West       26.094571
52         Bundoora - North       25.993778
162                Hillside       24.434810
157    Heidelberg - Rosanna       24.191940
